## Setup: Install Dependencies

Installs every dependency declared in `pyproject.toml` explicitly (accelerate, ai-edge-litert, datasets, litert-torch, torch, torchvision, transformers) using `%pip`, the conventional way to install packages from a notebook.

In [ ]:
# Install all project dependencies explicitly (from pyproject.toml)
%pip install -q "accelerate>=1.14.0" "ai-edge-litert>=2.2.0" "datasets>=5.0.1" "litert-torch>=0.9.4" "torch>=2.13.0" "torchvision>=0.28.0" "transformers>=5.16.1"

## Step 1: Extract SMS Records

Parses the raw `sms_dump.txt` backup by splitting on `Row:` markers, extracts each message's `body` (text) and `address`, and writes the unlabeled records to `data/sms_unlabeled.jsonl`.

SMS can be exported with the following command

```bash
adb shell content query --uri content://sms/ > ./data/sms_dump.txt
```

In [ ]:
# File paths

REGEX_RULES_FILE_PATH = "./data/regex_rules.json"
SMS_DUMP_FILE_PATH = "./data/sms_dump.txt"
SMS_UNLABELED_FILE_PATH = "./data/sms_unlabeled.jsonl"
SMS_LABELED_FILE_PATH = "./data/sms_labeled.jsonl"

In [ ]:
import re
import json

with open(SMS_DUMP_FILE_PATH, encoding="utf-8") as f:
    raw = f.read()

# Split on "Row:" markers, each row = one message
rows = re.split(r'\nRow: \d+ ', raw)

records = []
for row in rows:
    body_match = re.search(r'body=(.*?)(?=, service_center=)', row, re.DOTALL)
    address_match = re.search(r'address=([^,]*)', row)
    if body_match and address_match:
        records.append({
            "address": address_match.group(1).strip(),
            "text": body_match.group(1).strip(),
            "label": None  # fill in during labeling
        })

with open(SMS_UNLABELED_FILE_PATH, "w", encoding="utf-8") as f:
    for r in records:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Extracted {len(records)} messages")

## Step 2: Label SMS with Regex Rules

Loads the unlabeled SMS from `sms_unlabeled.jsonl`, applies the pre-compiled spam regex rules from `data/d/out.json` to each message (defaulting to `ham`, flagged `spam` on any match), and writes the labeled records to `data/sms_labeled.jsonl`.

In [ ]:
import re
import json

def main():
    # 1. Load rules
    with open(REGEX_RULES_FILE_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)

    # OPTIMIZATION: Pre-compile regex patterns for massive speedup
    # re.DOTALL ensures '.*' matches newline characters (\n) in the SMS text
    compiled_rules = [re.compile(rule['pattern'], re.DOTALL) for rule in data['rules']]
    print(compiled_rules)

    # 2. Load unlabeled SMS
    unlabeled_sms_labels = []
    with open(SMS_UNLABELED_FILE_PATH, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()
            if line:
                unlabeled_sms_labels.append(json.loads(line))

    print(f"Loaded {len(unlabeled_sms_labels)} records.")

    # 3. Process each SMS
    for sms_label in unlabeled_sms_labels:
        sms_label["label"] = "ham"  # Default assumption
        
        text = sms_label["text"]
        for pattern in compiled_rules:
            # re.search is generally safer than re.match for finding substrings
            if pattern.search(text):
                sms_label["label"] = "spam"
                break  # STOP checking other rules once a match is found

    # 4. Save the labeled data
    with open(SMS_LABELED_FILE_PATH, "w", encoding="utf-8") as f:
        for r in unlabeled_sms_labels:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
            
    print(f"Successfully saved labeled data to {SMS_LABELED_FILE_PATH}")

if __name__ == '__main__':
    main()

## Step 3: Test Tokenizer

Loads ParsBERT (bert-fa-zwnj-base) tokenizer and model, tokenizes a sample Persian SMS, and prints the token ids/output shape.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

model_name = "HooshvareLab/bert-fa-zwnj-base"  # ParsBERT v3, handles ZWNJ correctly
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

text = "تبریک میگم! شما برنده جایزه شدید، همین الان دریافت کنید"
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

print("//=========//")
print("//=========//")
print("//=========//")
print(tokenizer.convert_ids_to_tokens(inputs["input_ids"][0]))
print(outputs.last_hidden_state.shape)

## Step 4: Fine-tune ParsBERT

Reads labeled SMS data from `data/sms_labeled.jsonl`, cleans text, converts string labels to ints, builds a train/validation split, and fine-tunes a sequence-classification model (spam/ham).

In [ ]:
import json

from datasets import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

model_name = "HooshvareLab/bert-fa-zwnj-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name, 
    num_labels=2,
    id2label={0: "ham", 1: "spam"},
    label2id={"ham": 0, "spam": 1}
)

# Replace with your real logged/labeled data
texts = []
labels = []  # 1 = spam, 0 = ham

with open(SMS_LABELED_FILE_PATH, "r", encoding="utf-8") as file:
    for line in file:
        line = line.strip()
        if line:
            data = json.loads(line)
            
            # FIX 1: Clean newlines and extra spaces
            clean_text = data["text"].replace("\n", " ").replace("\r", " ").strip()
            texts.append(clean_text)
            
            # FIX 2: Convert string labels to integers (0 for ham, 1 for spam)
            label_int = 1 if data["label"].lower() == "spam" else 0
            labels.append(label_int)


# 3. Create Dataset and Split into Train/Validation (Best Practice)
full_dataset = Dataset.from_dict({"text": texts, "label": labels})
# Split 90% for training, 10% for validation to check for overfitting
split_dataset = full_dataset.train_test_split(test_size=0.1, seed=42)

def tokenize(batch):
    return tokenizer(batch["text"], padding="max_length", truncation=True, max_length=128)

# Apply tokenization
tokenized_datasets = split_dataset.map(tokenize, batched=True)
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])


training_args = TrainingArguments(
    output_dir="./parsbert_spam_model",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    logging_steps=10,
    save_strategy="epoch",

    per_device_eval_batch_size=8,
    eval_strategy="epoch", # Evaluate at the end of each epoch
    load_best_model_at_end=True, # Automatically keeps the best performing model
    metric_for_best_model="eval_loss"
)

trainer = Trainer(
    model=model, 
    args=training_args, 
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"] # Validation set
)
trainer.train()

model.save_pretrained("./parsbert_spam_final")
tokenizer.save_pretrained("./parsbert_spam_final")

## Step 5: Run Classifier

Loads the fine-tuned model from `./parsbert_spam_final` and interactively classifies text as SPAM or HAM with a spam-probability threshold.

In [ ]:
# from transformers import pipeline

# text = "سلام"
# text = "تخفیف"
# # text = "کد تخفیف ویژه شما آماده است، همین الان دریافت کنید"
# classifier = pipeline("text-classification", model="./parsbert_spam_final", tokenizer="./parsbert_spam_final")
# result = classifier(text)
# print(f"{text=}")
# print(result)

from transformers import pipeline

classifier = pipeline("text-classification", model="./parsbert_spam_final", tokenizer="./parsbert_spam_final")

def check_spam(text, threshold=0.5):
    result = classifier(text)[0]
    label, score = result["label"], result["score"]
    
    # Normalize to "probability of spam" regardless of which label won
    spam_prob = score if label == "spam" else (1 - score)
    
    is_spam = spam_prob >= threshold
    return spam_prob, is_spam

if __name__ == "__main__":
    print("Enter text to check (Ctrl+C to quit):")
    while True:
        text = input("> ")
        spam_prob, is_spam = check_spam(text)
        verdict = "SPAM" if is_spam else "HAM"
        print(f"{verdict} (spam probability: {spam_prob:.1%})")

## Step 6: Convert to TFLite

Converts the fine-tuned model to a TFLite (AI Edge Torch) model, sanity-checks outputs against the PyTorch model, and exports `parsbert_spam.tflite`.

In [ ]:
import litert_torch
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model_path = "./parsbert_spam_final"
model = AutoModelForSequenceClassification.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)
model.eval()

# Build a sample input matching your fine-tuning max_length
sample_text = "این یک متن نمونه است"
inputs = tokenizer(sample_text, return_tensors="pt", padding="max_length", truncation=True, max_length=128)
sample_inputs = (inputs["input_ids"], inputs["attention_mask"])

edge_model = litert_torch.convert(model, sample_inputs)

# Sanity check: compare PyTorch output vs converted output
import numpy as np

with torch.no_grad():
    torch_output = model(*sample_inputs).logits
edge_output = edge_model(*sample_inputs)

if isinstance(edge_output, dict):
    edge_logits = edge_output["logits"]
else:
    edge_logits = edge_output

if np.allclose(torch_output.numpy(), edge_logits, atol=1e-4):
    print("Conversion verified — outputs match")
else:
    print("Warning: outputs diverge, inspect before deploying")
    print("Torch:", torch_output.numpy())
    print("Edge:", edge_logits)

edge_model.export("parsbert_spam.tflite")